# ChuckleNet: WavLM-Base+ Utterance Embedding Extraction

Extract 768-dim WavLM embeddings for 15,060 utterances from stand-up comedy.

## Architecture
- **Model**: microsoft/wavlm-base-plus (768-dim)
- **Input**: Audio clips (full utterance, ~8s average)
- **Output**: 768-dim mean-pooled embedding per utterance
- **Time**: ~2-3 hours on T4 GPU for 15K utterances
- **Result**: `wavlm_utterance_embeddings.pt` (768 × 15,060 vectors)

## For Context
- Previous best audio-only: F1=0.30 (hand-crafted features)
- Previous best text-only: F1=0.82 (XLM-R word-level)
- **Target**: WavLM utterance-level F1=0.55-0.65, fusion F1>0.85

In [ ]:
#@title 1. Install dependencies
!pip install transformers torch torchaudio datasets soundfile -q
!nvidia-smi  # Check GPU

In [ ]:
#@title 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set paths
GDRIVE_BASE = '/content/drive/MyDrive/laughter_prediction_backup'
UTTERANCES_PATH = f'{GDRIVE_BASE}/aligned_utterances.jsonl'
AUDIO_DIR = f'{GDRIVE_BASE}/audio'
OUTPUT_PATH = f'{GDRIVE_BASE}/wavlm_utterance_embeddings.pt'

import os
assert os.path.exists(UTTERANCES_PATH), f'Utterances not found at {UTTERANCES_PATH}'
print(f'✅ Found utterances: {UTTERANCES_PATH}')
print(f'✅ Audio dir: {AUDIO_DIR}')

In [ ]:
#@title 3. Load utterances and find audio files
import json, glob

utterances = []
with open(UTTERANCES_PATH) as f:
    for line in f:
        utterances.append(json.loads(line))

print(f'Loaded {len(utterances)} utterances')
print(f'Positive: {sum(1 for u in utterances if u["label_any"])}')

# Find audio files for each video
video_to_audio = {}
for ext in ['*.mp3', '*.wav', '*.m4a', '*.opus']:
    for path in glob.glob(f'{AUDIO_DIR}/**/{ext}', recursive=True):
        vid = os.path.basename(path).rsplit('.', 1)[0]
        video_to_audio[vid] = path

print(f'Found {len(video_to_audio)} audio files')

# Check utterance-coverage
covered = sum(1 for u in utterances if u['video_id'] in video_to_audio)
print(f'Utterances with audio: {covered}/{len(utterances)} ({100*covered/len(utterances):.1f}%)')

In [ ]:
#@title 4. Load WavLM-Base+
import torch
from transformers import WavLMModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

wavlm = WavLMModel.from_pretrained('microsoft/wavlm-base-plus')
wavlm = wavlm.to(device)
wavlm.eval()
print(f'✅ WavLM-Base+ loaded ({sum(p.numel() for p in wavlm.parameters()):,} params)')

In [ ]:
#@title 5. Extract embeddings (main loop)
import torchaudio
import numpy as np
from tqdm.auto import tqdm

SAMPLE_RATE = 16000
CONTEXT_PAD = 0.2  # 200ms padding

embeddings = {}
errors = []
audio_cache = {}  # Cache loaded audio arrays

def load_audio(video_id):
    """Load and cache audio for a video."""
    if video_id in audio_cache:
        return audio_cache[video_id]
    path = video_to_audio.get(video_id)
    if path is None:
        return None, None
    try:
        wav, sr = torchaudio.load(path)
        if sr != SAMPLE_RATE:
            wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        wav = wav.squeeze(0)
        audio_cache[video_id] = (wav, SAMPLE_RATE)
        return wav, SAMPLE_RATE
    except Exception as e:
        errors.append((video_id, str(e)))
        return None, None

for i, utt in enumerate(tqdm(utterances, desc='Extracting WavLM embeddings')):
    uid = utt['utterance_id']
    vid = utt['video_id']
    
    # Load audio
    wav, sr = load_audio(vid)
    if wav is None:
        continue
    
    # Extract clip with context padding
    start_sample = max(0, int((utt['start'] - CONTEXT_PAD) * SAMPLE_RATE))
    end_sample = min(len(wav), int((utt['end'] + CONTEXT_PAD) * SAMPLE_RATE))
    clip = wav[start_sample:end_sample]
    
    if len(clip) < SAMPLE_RATE * 0.1:  # Skip clips < 100ms
        continue
    
    # WavLM forward pass
    with torch.no_grad():
        input_tensor = clip.unsqueeze(0).to(device)
        output = wavlm(input_tensor)
        # Mean pool over time dimension
        emb = output.last_hidden_state.mean(dim=1).squeeze(0).cpu()
    
    embeddings[uid] = emb
    
    # Clear cache periodically to manage memory
    if i > 0 and i % 2000 == 0:
        # Keep only recent audio in cache
        recent_vids = set(u['video_id'] for u in utterances[max(0,i-500):i+500])
        audio_cache = {k: v for k, v in audio_cache.items() if k in recent_vids}
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

print(f'\n✅ Extracted {len(embeddings)} embeddings')
print(f'❌ Errors: {len(errors)}')
if errors:
    for vid, err in errors[:5]:
        print(f'  {vid}: {err}')

In [ ]:
#@title 6. Save embeddings
import torch

# Convert to tensor
embedding_ids = sorted(embeddings.keys())
embedding_tensor = torch.stack([embeddings[uid] for uid in embedding_ids])

print(f'Embedding tensor shape: {embedding_tensor.shape}')
print(f'Expected: ({len(embedding_ids)}, 768)')

# Save as dict with metadata
save_dict = {
    'embeddings': embedding_tensor,
    'utterance_ids': embedding_ids,
    'model': 'microsoft/wavlm-base-plus',
    'pooling': 'mean',
    'context_padding': CONTEXT_PAD,
    'n_utterances': len(embedding_ids),
}

torch.save(save_dict, OUTPUT_PATH)
print(f'\n✅ Saved to {OUTPUT_PATH}')
print(f'   Size: {os.path.getsize(OUTPUT_PATH) / 1024 / 1024:.1f} MB')

In [ ]:
#@title 7. Verify saved embeddings
data = torch.load(OUTPUT_PATH)
print(f'Embeddings shape: {data["embeddings"].shape}')
print(f'Model: {data["model"]}')
print(f'N utterances: {data["n_utterances"]}')
print(f'Context padding: {data["context_padding"]}s')
print(f'Sample IDs: {data["utterance_ids"][:5]}')
print(f'\n✅ Ready for Phase 2 fusion training!')
print(f'Copy this file to: data/audio_comedy/wavlm_utterance_embeddings.pt')